In [1]:
import os
import json
import torch
import gc
import time
import sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv, find_dotenv

_env_file = find_dotenv()
REPO_DIR = Path(_env_file).parent.resolve()
load_dotenv(_env_file)
ROOT = (REPO_DIR / os.getenv("ROOT")).resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.experiment_notebook import setup_experiment, fine_tune_and_evaluate
from src.utils.utils import set_all_seeds
from src.fairness_losses import get_fairness_loss

In [2]:
experiment = ""  # "" = nou experiment | "20260414_153000" = reprendre un existent

In [3]:
# ── Configuració del sweep ──────────────────────────────────────────────────
N_RUNS          = 20                          # models per configuració
BASE_SEED       = 42                         # seeds: BASE_SEED, BASE_SEED+1, …

ALPHAS        = [1, 2.5, 5, 7.5, 10, 25, 50, 75, 100, 250, 500, 750, 1000]
LAYER_SIZES   = [2, 4, 6, 8]

MAX_EPOCHS    = 100   # màxim d'epochs per run
PATIENCE      = 10    # early stopping: atura si val_loss no millora en N epochs
BATCH_SIZE    = 32

print(f"Quantitat de runs: {len(ALPHAS) * len(LAYER_SIZES) * N_RUNS}")
FAIRNESS_LOSS = "tpr_parity"   # tpr_parity | demographic_parity | disentanglement
DATASET       = "NIH_Chest_XRAY_raw"  # subdirectori sota data/ (e.g. NIH_Chest_XRAY, CheXpert-v1.0-small_raw)
IMG_SUBDIR    = "sel_images"   # subdirectori d'imatges dins DATASET ("" si les imatges ja són a l'arrel)

# Atribut sensible: 'gender' | 'age_group' | 'view_pos_binary'
SENSITIVE_ATTR = "gender"
# Noms dels grups (valor 0 i valor 1 de l'atribut sensible)
GROUP_LABELS   = {0: "Male", 1: "Female"}
# Exemples per altres atributs:
#   SENSITIVE_ATTR = "age_group";       GROUP_LABELS = {0: "Young", 1: "Senior"}
#   SENSITIVE_ATTR = "view_pos_binary"; GROUP_LABELS = {0: "PA",    1: "AP"}

Quantitat de runs: 1040


In [4]:
experiment = "";  DATASET = "NIH_Chest_XRAY_raw";  IMG_SUBDIR = "sel_images"
SENSITIVE_ATTR = "age_group";  GROUP_LABELS = {0: "Young", 1: "Senior"}
ALPHAS = [1,2.5,5,7.5,10,25,50,75,100,250,500,750,1000]
LAYER_SIZES = [2,4,6,8];  N_RUNS = 20


In [5]:
# ── Gestió de l'experiment ──────────────────────────────────────────────────
RESULTS_BASE = REPO_DIR / "results" / "sweep_bias_accuracy"
RESULTS_BASE.mkdir(parents=True, exist_ok=True)

if experiment == "":
    EXP_NAME = datetime.now().strftime("%Y%m%d_%H%M%S")
    EXP_DIR = RESULTS_BASE / EXP_NAME
    EXP_DIR.mkdir(parents=True, exist_ok=True)

    config = {
        "N_RUNS": N_RUNS,
        "BASE_SEED": BASE_SEED,
        "ALPHAS": ALPHAS,
        "LAYER_SIZES": LAYER_SIZES,
        "MAX_EPOCHS": MAX_EPOCHS,
        "PATIENCE": PATIENCE,
        "BATCH_SIZE": BATCH_SIZE,
        "FAIRNESS_LOSS": FAIRNESS_LOSS,
        "DATASET": DATASET,
        "IMG_SUBDIR": IMG_SUBDIR,
        "SENSITIVE_ATTR": SENSITIVE_ATTR,
        "GROUP_LABELS": {str(k): v for k, v in GROUP_LABELS.items()},
    }
    with open(EXP_DIR / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    print(f"Nou experiment: {EXP_NAME}")

else:
    EXP_DIR = RESULTS_BASE / experiment
    if not EXP_DIR.exists():
        available = [d.name for d in RESULTS_BASE.iterdir() if d.is_dir()]
        raise FileNotFoundError(
            f"No s'ha trobat l'experiment '{experiment}' a {RESULTS_BASE}\n"
            f"Experiments disponibles: {available}"
        )
    with open(EXP_DIR / "config.json") as f:
        config = json.load(f)
    N_RUNS        = config["N_RUNS"]
    BASE_SEED     = config["BASE_SEED"]
    ALPHAS        = config["ALPHAS"]
    LAYER_SIZES   = config["LAYER_SIZES"]
    MAX_EPOCHS    = config["MAX_EPOCHS"]
    PATIENCE      = config["PATIENCE"]
    BATCH_SIZE    = config["BATCH_SIZE"]
    FAIRNESS_LOSS = config.get("FAIRNESS_LOSS", "tpr_parity")
    DATASET       = config.get("DATASET", "NIH_Chest_XRAY")
    IMG_SUBDIR    = config.get("IMG_SUBDIR", "sel_images")
    SENSITIVE_ATTR = config.get("SENSITIVE_ATTR", "gender")
    GROUP_LABELS   = {int(k): v for k, v in config.get("GROUP_LABELS", {"0": "Male", "1": "Female"}).items()}
    print(f"Reprenent experiment: {experiment}")
    print(f"Config carregada: {config}")

RESULTS_FILE = str(EXP_DIR / "sweep_results_v2.json")
print(f"\nDirectori : {EXP_DIR}")
print(f"Resultats : {RESULTS_FILE}")

Nou experiment: 20260611_195149

Directori : /home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/results/sweep_bias_accuracy/20260611_195149
Resultats : /home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/results/sweep_bias_accuracy/20260611_195149/sweep_results_v2.json


In [6]:
# ── Sweep principal ─────────────────────────────────────────────────────────
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, 'r') as f:
        all_results = json.load(f)
    print(f"Carregats {len(all_results)} runs previs de '{RESULTS_FILE}'")
else:
    all_results = []
    print("Fitxer de resultats nou.")

done = {(r['alpha'], r['ls'], r['run_id']) for r in all_results}

total_configs = len(ALPHAS) * len(LAYER_SIZES) * N_RUNS
remaining     = total_configs - len(done)
print(f"Total configuracions: {total_configs}  |  Pendents: {remaining}")

g0_name = GROUP_LABELS[0]
g1_name = GROUP_LABELS[1]

RETRY_WAIT_SECONDS = 15 * 60  # 15 minuts

for alpha in ALPHAS:
    for ls in LAYER_SIZES:
        for run_id in range(N_RUNS):
            if (alpha, ls, run_id) in done:
                print(f"[SKIP] alpha={alpha}, ls={ls}, run={run_id}")
                continue

            seed = BASE_SEED + run_id
            attempt = 0

            while True:
                attempt += 1
                print(f"\n{'='*60}")
                print(f"alpha={alpha}, ls={ls}, run={run_id}/{N_RUNS-1}, seed={seed}" +
                      (f"  [intent {attempt}]" if attempt > 1 else ""))
                print('='*60)

                try:
                    ctx = setup_experiment(
                        alpha=alpha,
                        prev2last_layer_size=ls,
                        batch_size=BATCH_SIZE,
                        seed=seed,
                        early_stopping_patience=PATIENCE,
                        run_name=f"sweep_a{alpha}_ls{ls}_r{run_id}",
                        results_base_dir=EXP_DIR / f"results_{DATASET}",
                        fairness_loss_fn=get_fairness_loss(FAIRNESS_LOSS),
                        data_subdir=f"data/{DATASET}",
                        img_subdir=IMG_SUBDIR,
                        sensitive_attr=SENSITIVE_ATTR,
                        group_labels=GROUP_LABELS,
                    )
                    set_all_seeds(seed)

                    result = fine_tune_and_evaluate(ctx, epochs=MAX_EPOCHS, alpha=alpha)

                    cm_all = result['All']['confusion_matrix']

                    cm_g0 = result[g0_name]['confusion_matrix']
                    tpr_g0 = cm_g0[1][1] / (cm_g0[1][1] + cm_g0[1][0]) if (cm_g0[1][1] + cm_g0[1][0]) > 0 else float('nan')

                    cm_g1 = result[g1_name]['confusion_matrix']
                    tpr_g1 = cm_g1[1][1] / (cm_g1[1][1] + cm_g1[1][0]) if (cm_g1[1][1] + cm_g1[1][0]) > 0 else float('nan')

                    tpr_gap = abs(tpr_g0 - tpr_g1)

                    gender_pred = result.get('gender_pred', {})
                    last_layer_key = sorted(
                        [k for k in gender_pred.keys() if k.startswith('layer_')],
                        key=lambda x: int(x.split('_')[1])
                    )[-1] if gender_pred else None
                    sensitive_auc_last = gender_pred[last_layer_key]['auc'] if last_layer_key else float('nan')

                    run_result = {
                        'alpha'                   : alpha,
                        'ls'                      : ls,
                        'run_id'                  : run_id,
                        'seed'                    : seed,
                        'epochs_trained'          : ctx['total_epochs'],
                        'accuracy'                : result['All']['accuracy'],
                        'auc_all'                 : result['All']['auc'],
                        f'tpr_{g0_name.lower()}'  : tpr_g0,
                        f'tpr_{g1_name.lower()}'  : tpr_g1,
                        'tpr_gap'                 : tpr_gap,
                        'sensitive_auc_last_layer': sensitive_auc_last,
                        'sensitive_attr'          : SENSITIVE_ATTR,
                        'results_dir'             : ctx['results_dir'],
                    }
                    all_results.append(run_result)
                    done.add((alpha, ls, run_id))

                    with open(RESULTS_FILE, 'w') as f:
                        json.dump(all_results, f, indent=2)
                    print(f"  Run guardat ({ctx['total_epochs']} epochs) a '{RESULTS_FILE}'")
                    break  # èxit: passa al següent run

                except Exception as e:
                    if 'out of memory' in str(e).lower():
                        print(f"  [OOM] CUDA sense memòria (intent {attempt}): {e}")
                        gc.collect()
                        torch.cuda.empty_cache()
                        print(f"  Esperant {RETRY_WAIT_SECONDS // 60} minuts abans de reintentar...")
                        time.sleep(RETRY_WAIT_SECONDS)
                    else:
                        print(f"  ERROR inesperat: {e}")
                        import traceback; traceback.print_exc()
                        break  # errors no-OOM no es reintenten

                finally:
                    if 'ctx' in dir():
                        del ctx
                    gc.collect()
                    torch.cuda.empty_cache()

print("\nSweep completat!")

Fitxer de resultats nou.
Total configuracions: 1040  |  Pendents: 1040

alpha=1, ls=2, run=0/19, seed=42
Device: cuda
Config: alpha=1, layer_size=2, batch_size=32, lr=0.0001


  [OOM] CUDA sense memòria (intent 1): CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

  Esperant 15 minuts abans de reintentar...



alpha=1, ls=2, run=0/19, seed=42  [intent 2]
Device: cuda
Config: alpha=1, layer_size=2, batch_size=32, lr=0.0001
  [OOM] CUDA sense memòria (intent 2): CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.



  Esperant 15 minuts abans de reintentar...



alpha=1, ls=2, run=0/19, seed=42  [intent 3]
Device: cuda
Config: alpha=1, layer_size=2, batch_size=32, lr=0.0001
  [OOM] CUDA sense memòria (intent 3): CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.



...

Traceback (most recent call last):
  File "/tmp/ipykernel_4171815/3387225906.py", line 39, in <module>
    ctx = setup_experiment(
          ^^^^^^^^^^^^^^^^^
  File "/home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/src/experiment_notebook.py", line 248, in setup_experiment
    model = FeatureExtractor(base_resnet, intermediate_size=cfg['prev2last_layer_size']).to(device)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jmassalle/TFM/venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1384, in to
    return self._apply(convert)
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/jmassalle/TFM/venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 934, in _apply
    module._apply(fn)
  File "/home/jmassalle/TFM/venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 965, in _apply
    param_applied = fn(param)
                    ^^^^^^^^^
  File "/home/jmassalle/TFM/venv/lib/pytho

  ERROR inesperat: CUDA error: CUDA-capable device(s) is/are busy or unavailable
Search for `cudaErrorDevicesUnavailable' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


alpha=1000, ls=8, run=19/19, seed=61
Device: cuda
Config: alpha=1000, layer_size=8, batch_size=32, lr=0.0001


Traceback (most recent call last):
  File "/tmp/ipykernel_4171815/3387225906.py", line 39, in <module>
    ctx = setup_experiment(
          ^^^^^^^^^^^^^^^^^
  File "/home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/src/experiment_notebook.py", line 248, in setup_experiment
    model = FeatureExtractor(base_resnet, intermediate_size=cfg['prev2last_layer_size']).to(device)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jmassalle/TFM/venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1384, in to
    return self._apply(convert)
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/jmassalle/TFM/venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 934, in _apply
    module._apply(fn)
  File "/home/jmassalle/TFM/venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 965, in _apply
    param_applied = fn(param)
                    ^^^^^^^^^
  File "/home/jmassalle/TFM/venv/lib/pytho

  ERROR inesperat: CUDA error: CUDA-capable device(s) is/are busy or unavailable
Search for `cudaErrorDevicesUnavailable' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


Sweep completat!


Traceback (most recent call last):
  File "/tmp/ipykernel_4171815/3387225906.py", line 39, in <module>
    ctx = setup_experiment(
          ^^^^^^^^^^^^^^^^^
  File "/home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/src/experiment_notebook.py", line 248, in setup_experiment
    model = FeatureExtractor(base_resnet, intermediate_size=cfg['prev2last_layer_size']).to(device)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jmassalle/TFM/venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1384, in to
    return self._apply(convert)
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/jmassalle/TFM/venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 934, in _apply
    module._apply(fn)
  File "/home/jmassalle/TFM/venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 965, in _apply
    param_applied = fn(param)
                    ^^^^^^^^^
  File "/home/jmassalle/TFM/venv/lib/pytho